<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_78_bare_metal_provisioning_parallel_storage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🏗️ **Module 78 — Bare-Metal Provisioning + Parallel Storage** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps


# Module 78 — Bare-Metal Provisioning + Parallel Storage

> The course assumed cloud everywhere — `terraform apply` → EKS cluster (M48), `kubectl get nodes` → eight A100s appear (M46). **The frontier doesn't run that way.** Every serious AI lab — Anthropic, OpenAI, xAI, Meta, Mistral — runs at least some of its training fleet on **bare-metal servers** in **co-located data centres** using **parallel filesystems** they themselves operate. This module is that world: the racks, the PXE boots, the Lustre mounts, the IPMI fan curves.
>
> You won't run a real MAAS controller from Colab, but every command here is what you'd type in front of an actual rack.

### What you'll cover
1. Bare-metal vs cloud — why frontier labs run both
2. **MAAS** + PXE boot — turning empty boxes into a cluster
3. **Redfish / IPMI / BMC** — the management plane
4. **cloud-init** — the de-facto post-install hook
5. **Parallel filesystems** — Lustre · WekaIO · BeeGFS · GPFS · DAOS · VAST
6. **Local NVMe + scratch tiers** — the hot path
7. **Object storage** — Ceph · MinIO · S3 — for checkpoints + datasets
8. **Slurm vs Kubernetes** for HPC + ML hybrid clusters
9. Day-2 ops — health, ECC, NVLink errors, firmware
10. The 2026 reference architecture — DGX SuperPOD, EOS, Stargate


## 1 · Bare-metal vs cloud — why both

| Dimension | Cloud (M48 Terraform → EKS) | Bare-metal |
|---|---|---|
| **Time to first node** | minutes | weeks of physical install |
| **Cost per H100/hr** | ~$2.50-5 on-demand; ~$1.50-2.50 reserved | ~$0.40-0.80 amortised over 3 yrs |
| **GPU availability** | rationed; long waits for H100/B200 in 2024-25 | you own them |
| **Networking** | best-effort 100/200 Gb | choose your own NDR/XDR IB |
| **Storage** | EBS, EFS, FSx Lustre — fast but $$$ | Lustre / WekaIO at $0.10/GB/yr ranges |
| **Ops burden** | very low | very high — you replace failed nodes, firmware, fans |

**The rule of thumb in 2026:** if you need **> 1024 GPUs for > 12 months**, bare-metal in your own (or co-lo) data centre is cheaper. Below that, cloud wins. Above that, you have a real **systems** team — most "AI startups" actually run on cloud and budget for the bill.

**The hybrid path** (what most labs actually do):
- Bare-metal in **2-4 co-los** for long-running training.
- **Cloud burst** for inference, experiments, and bursty fine-tunes.
- Same **k8s** API on both → same Helm charts (M47), same observability (M50, M51), same models flow through.

## 2 · MAAS — Metal as a Service

Canonical's open-source tool to turn empty racks into a cloud-shaped API.

```
   ┌────────────────────────────────────────────────────────────┐
   │  MAAS region controller (PostgreSQL + Python service)      │
   │   - inventory of every NIC / disk / BMC                    │
   │   - DHCP + PXE + TFTP for boot                             │
   │   - REST + CLI + web UI                                    │
   └─────────────────────────┬──────────────────────────────────┘
                             │
                ┌────────────┴────────────┐
                ▼                          ▼
   ┌───────────────────────┐    ┌───────────────────────┐
   │  Rack controller 1    │    │  Rack controller 2     │
   │  - serves DHCP + TFTP │    │  - serves DHCP + TFTP  │
   └─────────┬─────────────┘    └─────────┬──────────────┘
             │                              │
        ┌────┴────┬────┬────┬────┐    ┌────┴────┬────┬─────┐
        ▼         ▼    ▼    ▼    ▼    ▼         ▼    ▼     ▼
     node    node node node node   node    node node  node
     (bare)
```

The lifecycle:
1. **Enlist** — power on a new server; PXE-boot to the MAAS ephemeral kernel; it inventories every disk / NIC / BMC and reports back.
2. **Commission** — run hardware tests (memtest, disk health, NIC throughput, GPU presence). Failures get tagged.
3. **Ready** — the box sits in the pool, available to deploy.
4. **Deploy** — `maas <profile> machines allocate` + `deploy` writes an image (Ubuntu, RHEL, etc.) to local disk, sets up the network, runs cloud-init, then boots.
5. **Release** — when done, wipe and return to the pool.

In [ ]:
# Typical MAAS CLI workflow once you've added racks to the controller
maas_cli = '''
# Authenticate
$ maas login admin http://maas:5240/MAAS $API_KEY

# See pools / machines
$ maas admin machines read | jq '.[] | {hostname, status, power_state}'

# Tag the GPU nodes for kubeadm
$ maas admin tag create name=gpu-h100 comment="DGX H100 boxes"
$ maas admin tag update-nodes gpu-h100 add=node1,node2,node3,node4

# Allocate + deploy 4 of them with a Ubuntu 24.04 + user-data file
$ for h in node1 node2 node3 node4; do
    maas admin machine deploy $h \
        distro_series=noble \
        user_data="$(base64 < cloud-init.yaml)" \
        hwe_kernel=ga-24.04 ;
  done

# Power-off everything (Redfish / IPMI under the hood)
$ maas admin machines power-off filter='tags=gpu-h100'
'''
print(maas_cli)

**Alternatives** you'll meet:
- **Foreman + Katello** — Red Hat ecosystem, similar PXE + image story.
- **Tinkerbell** — CNCF, Kubernetes-native bare-metal provisioning (used by Equinix Metal, OpenStack Ironic alternatives).
- **Cluster Manager / xCAT / Bright** — HPC-flavoured (Slurm-heavy clusters).
- **OpenStack Ironic** — bare-metal as part of an OpenStack deployment.
- **Tarmak / Sidero Metal / Talos Linux** — k8s-first bare-metal stacks.

## 3 · Redfish / IPMI / BMC — the management plane

Every server has a **Baseboard Management Controller (BMC)** — a tiny computer that runs even when the server is powered off. It listens on a dedicated network port and gives you:

- **Power** — `power on`, `power off`, `power cycle`.
- **Console** — KVM-over-IP for BIOS-level debugging.
- **Sensors** — temperatures, fan RPM, voltages, PSU status.
- **Inventory** — model numbers, firmware versions.
- **Boot order** — force next boot to PXE.
- **Firmware update** — apply BIOS / BMC firmware blobs.

Two protocols speak to the BMC:

| Protocol | Notes |
|---|---|
| **IPMI** | the legacy spec — UDP, optional auth, lots of vendor quirks. `ipmitool` is the canonical client. |
| **Redfish** | the modern HTTPS REST + JSON spec. Same operations as IPMI but auditable, scriptable, secure. **The default in 2026.** |

```bash
# Power state
$ curl -k -u admin:pw https://bmc-node1/redfish/v1/Systems/1 | jq .PowerState

# Power cycle
$ curl -k -u admin:pw -X POST https://bmc-node1/redfish/v1/Systems/1/Actions/ComputerSystem.Reset \
       -H 'Content-Type: application/json' -d '{"ResetType":"ForceRestart"}'

# Read sensor (temperature) — over Redfish chassis API
$ curl -k -u admin:pw https://bmc-node1/redfish/v1/Chassis/1/Thermal | jq '.Temperatures[] | {Name, ReadingCelsius}'
```

**Security note.** BMCs are a juicy attack surface. Put them on an **isolated management VLAN**, behind a jump host, never on the public internet. Rotate default credentials at install time. Use **firmware-signed images** only.

## 4 · cloud-init — the post-install hook

PXE boots the kernel. The kernel mounts root. Then **cloud-init** runs `user_data` — a YAML file that customises the box on first boot.

```yaml
# cloud-init.yaml — bare-metal user_data file
#cloud-config

hostname: dgx-node-04
manage_etc_hosts: true

users:
  - name: ops
    sudo: ALL=(ALL) NOPASSWD:ALL
    ssh_authorized_keys: ["ssh-ed25519 AAAA…  ops@bastion"]

packages:
  - nvidia-driver-550
  - nvidia-fabricmanager-550
  - nvidia-container-toolkit
  - mlnx-ofed-all
  - chrony

write_files:
  - path: /etc/security/limits.d/99-hpc.conf
    content: |
      * soft nofile 1048576
      * hard nofile 1048576
      * soft memlock unlimited
      * hard memlock unlimited
  - path: /etc/modules-load.d/k8s.conf
    content: |
      br_netfilter
      overlay

runcmd:
  - systemctl enable --now nvidia-fabricmanager
  - mkdir -p /mnt/lustre && mount -t lustre 10.0.5.1@o2ib:/scratch /mnt/lustre
  - curl -fsSL https://get.k3s.io | INSTALL_K3S_VERSION="v1.30.6+k3s1" sh -s - agent \
        --server https://control-plane:6443 --token "$(cat /etc/k3s-join-token)"

ntp:
  enabled: true
  pools: ["time.example.com"]

bootcmd:
  - [sysctl, -w, kernel.numa_balancing=0]   # critical for ML training perf
```

**cloud-init is the bridge between provisioning and configuration management.** Anything bigger than this YAML — driver versions, fabric tuning, security baselines — belongs in **Ansible** (M49) or **Salt** running *after* cloud-init brings the box online.

## 5 · Parallel filesystems — the data layer that frontier training needs

A training cluster reads from a corpus that's bigger than any one node. Writes go to gigabyte checkpoints from every rank simultaneously. A single NFS server collapses under that load. **Parallel filesystems** (PFS) solve it.

```
                          ┌──────────── compute nodes ────────────┐
                          │   GPU0 GPU1 GPU2 GPU3   GPU4 ...      │
                          └─┬────┬────┬────┬─────────┬────────────┘
                            │    │    │    │         │  POSIX / NFS / mount
            ┌───────────────┼────┼────┼────┼─────────┼─────────────┐
            ▼               ▼    ▼    ▼    ▼         ▼             ▼
       ┌────────┐      ┌────────┐    ┌────────┐    ┌────────┐
       │ MDS    │      │ OSS 0  │    │ OSS 1  │    │ OSS 2  │  ← (Lustre roles)
       │ metadata│     │ object │    │ object │    │ object │
       └────────┘      └────────┘    └────────┘    └────────┘
                            │             │              │
                            ▼             ▼              ▼
                       (NVMe arrays, hundreds of TB each)
```

| Filesystem | Notes |
|---|---|
| **Lustre** | OSS standard; powers most TOP500 supercomputers; mature; complex to operate |
| **WekaIO** | commercial; NVMe-first; converged metadata + data; HPC-class numbers, simpler ops |
| **BeeGFS** | OSS; lighter than Lustre; popular in smaller HPC + ML shops |
| **IBM Spectrum Scale (GPFS)** | enterprise; mixed-workload heritage |
| **VAST Data** | proprietary; QLC-flash + DASE architecture; ~30 % of new AI clusters in 2024-25 |
| **DAOS** | Intel/HPE open-source; designed for storage-class memory + NVMe |
| **NetApp / Pure / DDN** | enterprise vendors with AI-optimised offerings |

**The 2026 honest field choice:**
- **Frontier AI lab (multi-thousand GPUs, in-house ops):** Lustre or DAOS.
- **Mid-size lab / many enterprises:** **WekaIO** or **VAST** — write a cheque, get HPC numbers + an enterprise GUI.
- **OSS-only / budget-constrained:** Lustre (mature) or BeeGFS (simpler).

### What good looks like
- **Read throughput** ≥ 1 TB/s aggregate per pod of compute.
- **Write throughput** ≥ 500 GB/s during checkpoint storms.
- **Small-file metadata ops** matter as much as bandwidth — millions of file opens during dataset enumeration.
- **POSIX semantics** so PyTorch / NCCL / Slurm don't have to be teach new tricks.

## 6 · Local NVMe — the hot tier

Even with Lustre at 1 TB/s, the cheapest fast storage is **local NVMe on every GPU node**. Two patterns:

| Pattern | Use |
|---|---|
| **Scratch** | per-job, per-node — `/scratch/$JOB_ID/` mounted as `tmpfs` or local NVMe. Wiped on exit. |
| **Local dataset cache** | every node caches the same shard of the dataset. The training script reads local; only first epoch touches the PFS. |
| **`OS_DISK` vs `DATA_DISK`** | put the kernel + Docker images on smaller SATA SSD; reserve the NVMe for training data + checkpoints. |

Modern H100/H200 nodes ship with **8-16× 3.84 TB NVMe** — that's ~30-60 TB of local fast storage **per node**. For datasets up to ~50 TB you can sidestep the PFS entirely.

## 7 · Object storage — checkpoints + dataset cold tier

Models like to write **big** checkpoints. Llama-3-405B in fp16 is ~810 GB. Saving every 1000 steps over a 6-month run = thousands of multi-hundred-GB checkpoints. Object storage handles it.

| Layer | What |
|---|---|
| **S3 / GCS / Azure Blob** | the default cloud answer. `s3a://` or `s3://` mountable from PyTorch / Spark. |
| **MinIO** | OSS, S3-compatible. Run on your own NVMe pool. |
| **Ceph (RGW)** | object-store front for the unified Ceph cluster. |
| **Pure FlashBlade / NetApp StorageGRID / IBM Cloud Object Storage** | enterprise on-prem |
| **VAST DataBase / Quobyte** | newer hybrid file + object |

### Tiering rule of thumb (frontier lab)
- **Local NVMe** — active dataset shard, last K checkpoints.
- **Parallel FS (Lustre / Weka)** — full active dataset.
- **Object storage (S3 / MinIO / Ceph)** — historical checkpoints, raw web data, eval suites.

You can run all three on the **same hardware** (NVMe-rich nodes) by partitioning roles: e.g. half the nodes contribute to Lustre OSS, other half host MinIO. Most labs don't — they let a vendor own the parallel-FS tier.

## 8 · Slurm vs Kubernetes — scheduling for HPC + ML

You have racks, fabric, parallel FS. Now **schedule jobs**.

| Property | **Slurm** | **Kubernetes (M46)** |
|---|---|---|
| Heritage | HPC, batch | cloud-native, services |
| Job shape | `sbatch script.sh`, multi-node MPI | pods, deployments, jobs, statefulsets |
| Resource model | `--gres=gpu:8 --ntasks-per-node=8` | `nvidia.com/gpu: 8` |
| Gang scheduling | default | requires plugin (Volcano, Kueue) |
| Multi-tenant | accounts, partitions, QoS | namespaces + quotas |
| Long-running inference | awkward | native (M44-M71) |
| Existing tooling | NCCL, PyTorch DDP all support `srun` | `torchrun`, `accelerate` |
| Operator headcount | typical HPC team | typical cloud-native team |

**The 2026 honest field setup:**
- **Pure-training HPC lab** → **Slurm**. Battle-tested for MPI / NCCL / RDMA. Decades of operator know-how.
- **Mixed training + inference platform** → **Kubernetes** + **Volcano** or **Kueue** for gang scheduling, NVIDIA's **GPU Operator** for the device plugin, **KubeRay** for Ray-based jobs.
- **Both** — common: Slurm for the long pre-train, k8s for everything else. They can share the same parallel FS.

## 9 · Day-2 ops — the silent killers

Hardware breaks. At 1000 GPUs and 5 9s per GPU-hour, expect **~1 GPU failure per day**. The list of things you must monitor:

| Subsystem | What to watch | Tool |
|---|---|---|
| **GPU** | ECC errors, NVLink errors, Xid messages, throttling | **DCGM** + `dcgm-exporter` (M50) |
| **NIC** | IB link errors, RoCE PFC pauses, dropped packets | `ibcheckerrors`, switch telemetry |
| **Storage** | OSS / MDS load, slow IO ratio, journal errors | filesystem-specific exporters |
| **BMC / sensors** | temperature, fan RPM, PSU faults | Redfish polling into Prometheus |
| **Firmware** | BMC, BIOS, NIC, GPU VBIOS — needs scheduled refresh | Ansible playbooks (M49) |
| **Power** | per-rack draw, transient excursions during all-reduce | PDU SNMP feeds |

The **canonical alerting setup**:
- DCGM → Prometheus (M50) → Grafana → PagerDuty.
- Slurm / k8s drain a node automatically on Xid errors above a threshold.
- Per-node `/var/log/nvidia-smi-events` archived to S3.

> 🟡 **Xid 79 (GPU has fallen off the bus)** is the classic frontier-training horror story. You're 14 days into a run, one GPU goes Xid 79, NCCL hangs all 1024 ranks until you intervene. Robust auto-restart with checkpoint-from-Lustre is the only defence.

## 10 · The 2026 reference architectures

What real frontier shops actually build looks like one of these:

### **NVIDIA DGX SuperPOD H100/H200 (reference)**
- Compute: 8× H100/H200 per node × 32-128 nodes per SU (Scalable Unit).
- Intra-node: NVSwitch (M77).
- Inter-node: NDR InfiniBand fat-tree, rail-optimised.
- Storage: customer choice — most pick **DDN AI400X** or **WEKA** or **VAST**.
- Mgmt: **Base Command Manager / Run:ai** + Slurm or k8s.

### **NVIDIA DGX GB200 NVL72 (2025+)**
- 72 Blackwells per NVL72 rack, fully NVLink-connected (~130 TB/s aggregate intra-rack).
- Liquid-cooled.
- Inter-rack: XDR IB at 800 Gb/s/port.
- Power: ~120 kW per rack (vs ~40 kW for an H100 SU).
- The new "single training unit" for frontier labs.

### **xAI Colossus (Memphis, 2024)**
- 100K+ H100 in a single liquid-cooled site, multiple buildings.
- Spectrum-X (RoCE Ethernet) — proves Ethernet can scale.

### **OpenAI Stargate (announced 2025)**
- ~$500B build-out across multiple US sites.
- 5+ GW of dedicated power; custom transformer substations.
- A reference point for "what an AI build looks like when energy is the bottleneck."

### Layers you'll see in any of them
1. **Building shell + power + cooling** — 15-100 MW per pod.
2. **Racks + servers + GPUs** — DGX or HGX OEM (Supermicro / Quanta / Foxconn).
3. **Fabric** — InfiniBand or RoCE (M77).
4. **Storage** — parallel FS + object.
5. **Provisioning** — MAAS / Foreman / Tinkerbell.
6. **Scheduler** — Slurm + k8s.
7. **MLOps** — your usual stack (M44 vLLM, M46-M51, M68-M73).

## ✅ Recap
- Bare-metal makes sense when **> 1024 GPUs for > 12 months**. Below that, cloud usually wins.
- **MAAS / Foreman / Tinkerbell** turn empty racks into a cloud-shaped API via PXE + cloud-init.
- **Redfish** is the modern BMC API; IPMI is the legacy one. Put BMCs on an isolated VLAN.
- **Parallel filesystems** (Lustre / WekaIO / VAST / BeeGFS / DAOS / GPFS) are required at frontier scale; **WekaIO / VAST** are the simpler-ops options.
- **Local NVMe + parallel FS + object storage** is the three-tier pattern.
- **Slurm for HPC** training; **k8s + Volcano/Kueue** for mixed training-and-inference clusters; many labs run both.
- Day-2 ops dominate ops cost — DCGM, Redfish telemetry, firmware refresh cadence, auto-drain on Xid errors.
- The **GB200 NVL72** is the 2025+ unit; the bottleneck above it is **electricity**, not compute.

Next: **M79 — Cost / FinOps for AI Platforms**.
